In [ ]:
# packages to install for this session
# scikit-learn, nltk, wordcloud, matplotlib, gensim
# import stuff
import polars as pl
import nltk
from collections import Counter

In [ ]:
# read data
# load cleaned data
import pickle 
with open("transcripts_speech_clean.pickle", "rb") as f:
    transcripts = pickle.load(f)

In [ ]:
# tokenize
# join speech by session and speaker, excluding non-participant speakers
# exclude reading out loud events which are not spontaneous talk.
# combine everything one speaker says in one session to single string
speaker_toks = (
    transcripts
    .filter(pl.col("speaker").is_in(["NA", "GROUP", "Researcher"]).not_())
    .filter(pl.col("event").is_in(["Reading Story", "Reading Poem"]).not_() | pl.col("event").is_null())
    .group_by(["session", "speaker"])
    .agg(pl.col("speech").str.join(" ").alias("speech"))
    .with_columns(
        pl.col("speech")
        .map_elements( # lowercasing tokenizing and returning everything that is an alphabetic character (no punctuation)
            lambda t: [w for w in nltk.word_tokenize(t.lower()) if w.isalpha()], return_dtype=pl.List(pl.Utf8))
        .alias("tokens")
    )
)

In [ ]:
# Type-Token Ratio (TTR): number of unique words divided by total words.
# -> higher TTR indicates greater lexical diversity.
# note! ttr is length-dependent - speakers with fewer total words tend to score higher

ttr = speaker_toks.with_columns(
    pl.col("tokens").map_elements(
        lambda t: len(set(t)) / len(t) if len(t) > 0 else 0.0,
        return_dtype=pl.Float64
    ).alias("ttr")
).select(["session", "speaker", "ttr"]).sort(["session", "ttr"], descending=True)
ttr

In [ ]:
# bar chart of ttr per speaker and session

import altair as alt
ttr_dev = alt.Chart(ttr).mark_bar().encode( 
    x=alt.X("speaker:N"), 
    y=alt.Y("ttr:Q"), 
    color="session:N", 
    xOffset="session:N" 
)

ttr_dev

In [ ]:
# https://lexicalrichness.readthedocs.io/en/latest/ - mattr more complex than ttr here

In [ ]:
# word frequencies


from nltk.corpus import stopwords
nltk.download("stopwords")
stop = set(stopwords.words("english"))
stop



In [ ]:
# Pipeline:
#   1. same filtering as speaker_toks, but all speakers in the session combined
#   2. tokenise 
#   3. extract top 20 words and counts
#   4. remove standard English stopwords -> tokens_clean
#   5. extract top 20 words and counts from cleaned tokens

session_toks = (
    transcripts
    .filter(pl.col("speaker").is_in(["NA", "GROUP", "Researcher"]).not_())
    .filter(pl.col("event").is_in(["Reading Story", "Reading Poem"]).not_() | pl.col("event").is_null())
    .group_by("session")
    .agg(pl.col("speech").str.join(" ").alias("speech"))
    .with_columns(
        pl.col("speech")
        .map_elements( # tokenizing and returning everything that is an alphabetic character (no punctuation)
            lambda t: [w for w in nltk.word_tokenize(t.lower()) if w.isalpha()], return_dtype=pl.List(pl.Utf8))
        .alias("tokens")
    )
    .with_columns(
        pl.col("tokens")
        .map_elements(
            lambda t: [w for w, _ in Counter(t).most_common(20)],
            return_dtype=pl.List(pl.Utf8)
        ).alias("top20_words"),
        pl.col("tokens")
        .map_elements(
            lambda t: [c for _, c in Counter(t).most_common(20)],
            return_dtype=pl.List(pl.Int32)
        ).alias("top20_counts")
    )
    .with_columns(
        pl.col("tokens")
        .map_elements(
            lambda t: [w for w in t if w not in stop],
            return_dtype=pl.List(pl.Utf8)
        )
        .alias("tokens_clean")
    )
    .with_columns(
        pl.col("tokens_clean")
        .map_elements(
            lambda t: [w for w, _ in Counter(t).most_common(20)],
            return_dtype=pl.List(pl.Utf8)
        ).alias("top20_words_clean"),
        pl.col("tokens_clean")
        .map_elements(
            lambda t: [c for _, c in Counter(t).most_common(20)],
            return_dtype=pl.List(pl.Int32)
        ).alias("top20_counts_clean")
    )
)

In [ ]:
# most frequent words per session before stopword removal
# across sessions — we care about internal ranking, not absolute counts.
mfws_per_session = (
    session_toks
    .select(["session", "top20_words", "top20_counts"])
    .explode(["top20_words", "top20_counts"])
)
chart = alt.Chart(mfws_per_session).mark_bar().encode(
    y=alt.Y("top20_words:N", sort="-x", title="Word"),
    x=alt.X("top20_counts:Q", title="Frequency"),
    facet=alt.Facet("session:N", columns=2)
).properties(width=300, height=200).resolve_scale(y="independent") # each session gets own y-axis

chart.show()

In [ ]:
# Same chart after applying stopwords.

mfws_per_session = (
    session_toks
    .select(["session", "top20_words_clean", "top20_counts_clean"])
    .explode(["top20_words_clean", "top20_counts_clean"])
)
chart = alt.Chart(mfws_per_session).mark_bar().encode(
    y=alt.Y("top20_words_clean:N", sort="-x", title="Word"),
    x=alt.X("top20_counts_clean:Q", title="Frequency"),
    facet=alt.Facet("session:N", columns=2)
).properties(width=300, height=200).resolve_scale(y="independent")

chart.show()

In [ ]:
# filter some words maybe?
type(stop)


In [ ]:
# extend stopword list with fillers and noisy words
# -> depends on RQ! I want to know the topic!
# this stopword lists is prob. built for written text

stop.update(["like", "yeah", "um", "uh", "mm", "hmm", "mmm", "oh", "something", "kind", "think", "feel", "know", "mean", "really", "well"])

# recompute with the updated list.

session_toks = session_toks.with_columns(
        pl.col("tokens")
        .map_elements(
            lambda t: [w for w in t if w not in stop],
            return_dtype=pl.List(pl.Utf8)
        )
        .alias("tokens_clean")
    ).sort("session")

In [ ]:
# well, I'm too lazy to do the 20 words thing again, so wordclouds!
from wordcloud import WordCloud
from matplotlib import pyplot as plt
# the wordcloud package comes with it's own tokenization function and stopwords
# but we have already done that
for row in session_toks.iter_rows(named=True):
    # we already have counted the tokens, but the next funcition expects a dictionary
    freq_dict = Counter(row["tokens_clean"]) 
    wordcloud = WordCloud().generate_from_frequencies(freq_dict)
    plt.figure()
    plt.title(row["session"])
    plt.imshow(wordcloud)
    plt.axis("off")
    plt.show()

In [ ]:
# tf-idf - remember?
# First TF-IDF attempt: fit on raw joined speech, one document per session.
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(smooth_idf=False, use_idf=True)
tfidf_matrix = vectorizer.fit_transform(session_toks["speech"].to_list())


In [ ]:
# Convert the sparse TF-IDF matrix to a Polars DataFrame with words as columns
# and one row per session. Session labels are added as an identifier column.
feature_names = vectorizer.get_feature_names_out()
sessions = session_toks["session"].to_list()

tfidf_df = pl.DataFrame(
    tfidf_matrix.toarray(),
    schema=feature_names.tolist()
).with_columns(
    pl.Series("session", sessions)
)

In [ ]:
# Corpus-level view: mean TF-IDF score per word across all sessions.
# high-scoring words should be locally frequent but not shared across sessions —

tfidf_corpus = (
    tfidf_df
    .select(pl.exclude("session"))
    .mean()
    .transpose(include_header=True, header_name="word", column_names=["tfidf_mean"])
    .sort("tfidf_mean", descending=True)
)
tfidf_corpus

In [ ]:
# Second attempt: refit on individual turns rather than whole sessions.
# More documents = more meaningful IDF scores?
# Reading events are excluded as before

transcripts_nr = transcripts.filter(pl.col("event").is_in(["Reading Story", "Reading Poem"]).not_() | pl.col("event").is_null())

tfidf_turns = vectorizer.fit_transform(transcripts_nr["speech"].to_list())

In [ ]:
# Corpus-level mean TF-IDF using turns as documents - but still.. 
feature_names = vectorizer.get_feature_names_out()

tfidf_corpus = (
    pl.DataFrame(tfidf_turns.toarray(), schema=feature_names.tolist())
    .mean()
    .transpose(include_header=True, header_name="word", column_names=["tfidf_mean"])
    .sort("tfidf_mean", descending=True)
)
tfidf_corpus

In [ ]:
# Sanity check: what proportion of turns contain "yeah"?
# If this is close to 1.0, "yeah" should have near-zero IDF —
transcripts_nr.filter(pl.col("speech").str.contains("yeah")).height / transcripts_nr.height

In [ ]:
# proportion of sessions
session_toks.filter(pl.col("speech").str.contains("yeah")).height / session_toks.height

In [ ]:
from gensim import corpora
from gensim.models import TfidfModel

# Build a dictionary from your cleaned tokens
dictionary = corpora.Dictionary(session_toks["tokens"].to_list())

# Convert to bag-of-words
corpus = [dictionary.doc2bow(tokens) for tokens in session_toks["tokens"].to_list()]

# Fit TF-IDF 
tfidf = TfidfModel(corpus)
corpus_tfidf = tfidf[corpus]

In [ ]:
# Inspect the top scoring (most distinctive) words per session.

for i, doc in enumerate(corpus_tfidf):
    print(f"\n{session_toks['session'][i]}")
    top = sorted(doc, key=lambda x: x[1], reverse=True)[:10] # sort by score descending, take top 10
    for word_id, score in top:
        print(f"  {dictionary[word_id]}: {score:.3f}") # dictionary[word_id] converts id back to token

In [ ]:
for i, doc in enumerate(corpus_tfidf):
    print(f"\n{session_toks['session'][i]}")
    bottom = sorted(doc, key=lambda x: x[1])[:10] # sort ascending — lowest non-zero scores first
    for word_id, score in bottom:
        print(f"  {dictionary[word_id]}: {score:.3f}")

In [ ]:
yeah_id = dictionary.token2id.get("yeah")  # look up "yeah"'s integer id, returns None if not in vocabulary
for i, doc in enumerate(corpus_tfidf):
    scores = dict(doc)  # convert [(word_id, score), ...] to {word_id: score} for fast lookup
    print(f"{session_toks['session'][i]}: {scores.get(yeah_id, 0):.3f}")  # default 0 if dropped from sparse output

In [ ]:
# collect all word ids that have a non-zero score in at least one document
nonzero_ids = set()
for doc in corpus_tfidf:
    nonzero_ids.update(word_id for word_id, score in doc) # collect every id that has a non-zero score anywhere

# everything else scored zero in every document
zero_ids = set(dictionary.token2id.values()) - nonzero_ids  # full vocabulary minus non-zero ids = universal words
zero_words = [dictionary[i] for i in zero_ids] # convert ids back to tokens
print(sorted(zero_words))